# Airthings API Exploration

Exploring Airthings API authentication, device metadata, and air quality measurements using Python.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

import requests
from dotenv import load_dotenv


load_dotenv()

CLIENT_ID = os.getenv("AIRTHINGS_CLIENT_ID")
CLIENT_SECRET = os.getenv("AIRTHINGS_CLIENT_SECRET")

TOKEN_URL = "https://accounts-api.airthings.com/v1/token"
DEVICES_URL = "https://ext-api.airthings.com/v1/devices"

RAW_DATA_DIR = Path("data/raw")


def get_access_token():
    payload = {
        "grant_type": "client_credentials",
        "scope": "read:device:current_values",
    }

    response = requests.post(
        TOKEN_URL,
        data=payload,
        auth=(CLIENT_ID, CLIENT_SECRET),
    )

    response.raise_for_status()
    return response.json()["access_token"]


def get_devices(access_token):
    headers = {"Authorization": f"Bearer {access_token}"}

    response = requests.get(
        DEVICES_URL,
        headers=headers,
    )

    response.raise_for_status()
    return response.json()


def save_json(data, filename_prefix):
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = RAW_DATA_DIR / f"{filename_prefix}_{timestamp}.json"

    with open(file_path, "w") as file:
        json.dump(data, file, indent=2)

    return file_path


def get_current_values(access_token, device_id):
    headers = {"Authorization": f"Bearer {access_token}"}

    url = f"https://ext-api.airthings.com/v1/devices/{device_id}/latest-samples"

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.json()


if __name__ == "__main__":
    token = get_access_token()
    devices = get_devices(token)

    saved_path = save_json(devices, "devices")

    print("Success!")
    print(f"Saved response to: {saved_path}")
    print(json.dumps(devices, indent=2))

Success!
Saved response to: data/raw/devices_20260516_170511.json
{
  "devices": [
    {
      "id": "2960154462",
      "deviceType": "VIEW_PLUS",
      "sensors": [
        "radonShortTermAvg",
        "temp",
        "humidity",
        "pressure",
        "co2",
        "voc",
        "pm1",
        "pm25"
      ],
      "segment": {
        "id": "ec023d06-b965-4f9c-b165-02f46b195d0a",
        "name": "Bedroom - North wall",
        "started": "2026-04-11T23:17:07",
        "active": true
      },
      "location": {
        "id": "6d4b48d4-39a2-4a94-bfed-8b33a047b9c5",
        "name": "My Home"
      },
      "productName": "View Plus"
    }
  ],
  "offset": 0
}


In [5]:
import json
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw")

list(raw_path.glob("*"))

[PosixPath('../data/raw/.gitkeep')]

In [ ]:
pd.json_normalize(devices)